# Model Results Comparison

This notebook loads saved model evaluation results and visualises model performance across feature sets. It expects the pipeline output file `model_results.csv`, usually produced by `main.py` under `output/results/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULT_CANDIDATES = [
    ROOT / "output" / "results" / "model_results.csv",
    ROOT / "notebooks" / "output" / "results" / "model_results.csv",
]

RESULT_PATH = next((path for path in RESULT_CANDIDATES if path.exists()), None)
RESULT_PATH

In [ ]:
if RESULT_PATH is None:
    searched = "\n".join(f"- {path}" for path in RESULT_CANDIDATES)
    raise FileNotFoundError(
        "Could not find model_results.csv. Run the training pipeline first, for example:\n"
        "    python main.py --skip-deep\n\n"
        f"Searched:\n{searched}"
    )

results = pd.read_csv(RESULT_PATH)
results

## Best Models

Lower RMSE, MAE, and MARD are better. Higher R² is better.

In [ ]:
metric_cols = [col for col in ["test_rmse", "test_mae", "test_mard", "test_r2", "cv_rmse", "cv_r2"] if col in results.columns]
ranked = results.sort_values("test_rmse").reset_index(drop=True)
ranked[["model", "features", *metric_cols]].style.format(precision=4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_df = ranked.copy()
plot_df["model_features"] = plot_df["model"] + " | " + plot_df["features"]

sns.barplot(data=plot_df, y="model_features", x="test_rmse", hue="features", dodge=False, ax=ax)
ax.set_title("Test RMSE by Model")
ax.set_xlabel("RMSE (mg/dL)")
ax.set_ylabel("")
ax.legend(title="Feature set", loc="lower right")
plt.tight_layout()

## Metric Heatmaps

In [ ]:
heatmap_metrics = [col for col in ["test_rmse", "test_mae", "test_mard", "test_r2"] if col in results.columns]

fig, axes = plt.subplots(1, len(heatmap_metrics), figsize=(5 * len(heatmap_metrics), 5), squeeze=False)
for ax, metric in zip(axes.ravel(), heatmap_metrics):
    pivot = results.pivot(index="model", columns="features", values=metric)
    cmap = "viridis_r" if metric != "test_r2" else "viridis"
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap=cmap, ax=ax)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.set_ylabel("")

plt.tight_layout()

## Feature Impact

This compares `with features` against `baseline` for each model. Negative ΔRMSE means the engineered features improved RMSE.

In [ ]:
required_feature_sets = {"baseline", "with features"}
feature_impact = []

for model, group in results.groupby("model"):
    feature_sets = set(group["features"])
    if not required_feature_sets.issubset(feature_sets):
        continue
    base = group[group["features"] == "baseline"].iloc[0]
    full = group[group["features"] == "with features"].iloc[0]
    row = {
        "model": model,
        "delta_rmse": full["test_rmse"] - base["test_rmse"],
    }
    if "test_r2" in results.columns:
        row["delta_r2"] = full["test_r2"] - base["test_r2"]
    feature_impact.append(row)

impact_df = pd.DataFrame(feature_impact).sort_values("delta_rmse")
impact_df

In [ ]:
if not impact_df.empty:
    fig, axes = plt.subplots(1, 2 if "delta_r2" in impact_df.columns else 1, figsize=(13, 5), squeeze=False)
    ax = axes.ravel()[0]
    colors = impact_df["delta_rmse"].map(lambda value: "#2E7D32" if value < 0 else "#C62828")
    ax.barh(impact_df["model"], impact_df["delta_rmse"], color=colors)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Feature Impact on RMSE")
    ax.set_xlabel("Δ RMSE: with features - baseline")
    ax.set_ylabel("")

    if "delta_r2" in impact_df.columns:
        ax = axes.ravel()[1]
        colors = impact_df["delta_r2"].map(lambda value: "#2E7D32" if value > 0 else "#C62828")
        ax.barh(impact_df["model"], impact_df["delta_r2"], color=colors)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title("Feature Impact on R²")
        ax.set_xlabel("Δ R²: with features - baseline")
        ax.set_ylabel("")

    plt.tight_layout()
else:
    print("Feature impact needs both 'baseline' and 'with features' rows for each model.")

## Cross-Validation vs Test Performance

In [ ]:
if {"cv_rmse", "test_rmse"}.issubset(results.columns):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=results, x="cv_rmse", y="test_rmse", hue="model", style="features", s=120, ax=ax)
    lo = min(results["cv_rmse"].min(), results["test_rmse"].min())
    hi = max(results["cv_rmse"].max(), results["test_rmse"].max())
    ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1)
    ax.set_title("CV RMSE vs Test RMSE")
    ax.set_xlabel("CV RMSE")
    ax.set_ylabel("Test RMSE")
    plt.tight_layout()
else:
    print("CV RMSE and test RMSE columns are required for this plot.")

## Save Figures

Use this optional cell to save a compact model comparison summary as a PNG.

In [ ]:
save_dir = ROOT / "output" / "plots"
save_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_df, y="model_features", x="test_rmse", hue="features", dodge=False, ax=ax)
ax.set_title("Model Comparison by Test RMSE")
ax.set_xlabel("RMSE (mg/dL)")
ax.set_ylabel("")
ax.legend(title="Feature set", loc="lower right")
plt.tight_layout()

summary_path = save_dir / "model_results_comparison.png"
fig.savefig(summary_path, dpi=200, bbox_inches="tight")
summary_path